# 0400 · BASE FINAL Y PRIMER MODELADO (temporal + espacial)

**Proyecto:** Predicción de tiempos de entrega e incumplimiento de SLA · SERVIENVIOS Bogotá
**Requisito previo:** haber corrido el notebook **0300_CleanTransformParquet** completo (genera `LOG_CONSOLIDADO/LOG_BASE_BOGOTA.parquet`), con la columna `G_NUMERO_GUIA` activada en la celda 29.

Este notebook hace cuatro cosas, en orden:
1. **Diagnóstico**: qué procesos aparecen como último movimiento y cuántas rutas tienen tiempo comercial.
2. **Base final**: cruce con dimensiones + variables objetivo (tiempo de entrega e incumplimiento) + variables temporales → `BASE_FINAL_BOGOTA.parquet`.
3. **Resumen y EDA rápido**: las cifras para la presentación (tasa real de incumplimiento, tiempos, por mes, franja y zona).
4. **Modelado**: LightGBM vs. línea base (lineal / logística) en tres escenarios de validación: aleatorio, temporal y temporal + espacial.

In [ ]:
# CELDA 1 · Librerías (en Databricks; si el clúster ya las tiene, no pasa nada)
%pip install duckdb lightgbm holidays

In [ ]:
# CELDA 2 · Importaciones y rutas (mismas rutas que el 0300)
import datetime
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

horainicio = datetime.datetime.now()

PATH_DATABRICKS = "/Volumes/proyectogrado/source/sourcedb/"

RUTA_TABLA_BASE   = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/LOG_BASE_BOGOTA.parquet"      # sale del 0300
RUTA_BASE_FINAL   = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/BASE_FINAL_BOGOTA.parquet"    # la crea este notebook
RUTA_RESULTADOS   = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/RESULTADOS_MODELOS.csv"       # la crea este notebook

PAR = f"{PATH_DATABRICKS}PARAMETRICAS/"
Archivo_PAR_REGIONALES_PAIS      = f"{PAR}PAR.REGIONALES_PAIS.parquet"
Archivo_PAR_CIUDADES             = f"{PAR}PAR.CIUDADES.parquet"
Archivo_PAR_PROCESOS             = f"{PAR}PAR.PROCESOS.parquet"
Archivo_PAR_ZONASURBANAS         = f"{PAR}PAR.ZONASURBANAS.parquet"
Archivo_PAR_RED_OPERATIVA_OFERTA = f"{PAR}PAR.RED_OPERATIVA_OFERTA.parquet"

print("Tabla base:", RUTA_TABLA_BASE)

## 1. Diagnóstico antes de calcular las variables objetivo

**Por qué:** el 0300 se queda con el *último* movimiento de cada guía entre los procesos 5, 6 y 9. Hay que confirmar cuál de esos procesos es la **entrega efectiva** y cuántos envíos tienen tiempo comercial en la red operativa.

In [ ]:
# CELDA 3 · ¿Qué proceso quedó como último movimiento?
df_procesos = duckdb.sql(f"""
    SELECT B.SEG_ID_PROCESO, P.NOMBRE_PROCESO_LOG, COUNT(*) AS ENVIOS,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PORCENTAJE
    FROM read_parquet('{RUTA_TABLA_BASE}') B
    LEFT JOIN read_parquet('{Archivo_PAR_PROCESOS}') P ON B.SEG_ID_PROCESO = P.ID_PROCESO
    GROUP BY ALL ORDER BY ENVIOS DESC
""").df()
display(df_procesos)

In [ ]:
# CELDA 4 · CONFIGURACIÓN — ajustar según la salida de la celda 3
# Código(s) de ID_PROCESO que significan "entregado al destinatario".
ID_PROCESO_ENTREGA = [9]

# Mes de corte para la partición temporal: se entrena con meses <= MES_CORTE y se prueba con los posteriores.
# Con datos de enero a mayo: 3 => entrenar ene–mar, probar abr–may.
MES_CORTE = 3

# H2: False = solo variables internas | True = internas + clima (requiere haber corrido el 0150).
# Corran el modelado dos veces (False y True) y comparen las tablas de la celda 12.
USAR_CLIMA = False

ids_entrega_sql = ",".join(str(i) for i in ID_PROCESO_ENTREGA)
print("Proceso de entrega:", ID_PROCESO_ENTREGA, "| Mes de corte:", MES_CORTE, "| Clima:", USAR_CLIMA)

In [ ]:
# CELDA 5 · Cobertura de la red operativa (¿cuántos envíos tienen tiempo comercial?)
df_cobertura = duckdb.sql(f"""
    WITH RED AS (
        SELECT ID_CIUDAD_ORIGEN, ID_CIUDAD_DESTINO, ID_TIEMPO, ID_MEDTRANS,
               MIN(TIEMPO_ENTREGA_COMERCIAL_HOR) AS TIEMPO_COMERCIAL_HOR
        FROM read_parquet('{Archivo_PAR_RED_OPERATIVA_OFERTA}')
        GROUP BY ALL
    )
    SELECT
        COUNT(*) AS ENVIOS_ENTREGADOS,
        SUM(CASE WHEN R.TIEMPO_COMERCIAL_HOR IS NOT NULL THEN 1 ELSE 0 END) AS CON_TIEMPO_COMERCIAL,
        ROUND(100.0 * SUM(CASE WHEN R.TIEMPO_COMERCIAL_HOR IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS PCT_COBERTURA
    FROM read_parquet('{RUTA_TABLA_BASE}') B
    LEFT JOIN RED R
      ON B.G_ID_CIUDAD_ORIGEN = R.ID_CIUDAD_ORIGEN
     AND B.G_ID_CIUDAD_DESTINO = R.ID_CIUDAD_DESTINO
     AND COALESCE(NULLIF(B.G_ID_TIEMPO, 0), 1)   = R.ID_TIEMPO
     AND COALESCE(NULLIF(B.G_ID_MEDTRANS, 0), 1) = R.ID_MEDTRANS
    WHERE B.SEG_ID_PROCESO IN ({ids_entrega_sql})
""").df()
display(df_cobertura)
print("Si PCT_COBERTURA es bajo, revisar la llave de cruce con la red operativa antes de seguir.")

## 2. Base final: dimensiones + variables objetivo + variables temporales

- **TIEMPO_ENTREGA_HORAS** = horas entre la fecha de envío de la guía y el movimiento de entrega.
- **INCUMPLIMIENTO_SLA** = 1 si ese tiempo supera el tiempo comercial de la ruta (red operativa).
- Se excluyen tiempos negativos o mayores a 30 días (720 h) y envíos sin tiempo comercial.
- La red operativa se agrega por llave (MIN) para que un envío no se duplique si la tabla tiene filas repetidas.

In [ ]:
# CELDA 6 · Construcción de BASE_FINAL_BOGOTA.parquet
con = duckdb.connect()
con.execute(f"""
COPY (
    WITH RED AS (
        SELECT ID_CIUDAD_ORIGEN, ID_CIUDAD_DESTINO, ID_TIEMPO, ID_MEDTRANS,
               MIN(TIEMPO_ENTREGA_COMERCIAL_HOR) AS TIEMPO_COMERCIAL_HOR
        FROM read_parquet('{Archivo_PAR_RED_OPERATIVA_OFERTA}')
        GROUP BY ALL
    ),
    ZON AS (
        SELECT ID_PAIS, ID_CIUDAD, ID_TIPO_ZONA_URBA, ID_ZONA_URBA,
               ANY_VALUE(NOMBRE_ZONA_URBA) AS NOMBRE_ZONA_URBA,
               ANY_VALUE(CATEGORIA_ZONA_URBA) AS CATEGORIA_ZONA_URBA
        FROM read_parquet('{Archivo_PAR_ZONASURBANAS}')
        GROUP BY ALL
    ),
    BASE AS (
        SELECT
            B.*,
            R.TIEMPO_COMERCIAL_HOR,
            date_diff('minute', B.G_FECHA_ENVIO, B.SEG_FECHA_HORA_MOVIMIENTO) / 60.0 AS TIEMPO_ENTREGA_HORAS,
            CO.NOMBRE_CIUDAD     AS CIUDAD_ORIGEN,
            RO.NOMBRE_REGIONAL   AS REGIONAL_ORIGEN,
            Z.NOMBRE_ZONA_URBA   AS ZONA_URBANA,
            Z.CATEGORIA_ZONA_URBA AS CATEGORIA_ZONA,
            CAST(B.DOC_ID_TIPO_ZONA_URBA AS VARCHAR) || '-' || CAST(B.DOC_ID_ZONA_URBA AS VARCHAR) AS ZONA_ID
        FROM read_parquet('{RUTA_TABLA_BASE}') B
        LEFT JOIN RED R
          ON B.G_ID_CIUDAD_ORIGEN = R.ID_CIUDAD_ORIGEN
         AND B.G_ID_CIUDAD_DESTINO = R.ID_CIUDAD_DESTINO
         AND COALESCE(NULLIF(B.G_ID_TIEMPO, 0), 1)   = R.ID_TIEMPO
         AND COALESCE(NULLIF(B.G_ID_MEDTRANS, 0), 1) = R.ID_MEDTRANS
        LEFT JOIN read_parquet('{Archivo_PAR_CIUDADES}') CO
          ON B.G_ID_PAIS_ORIGEN = CO.ID_PAIS AND B.G_ID_CIUDAD_ORIGEN = CO.ID_CIUDAD
        LEFT JOIN read_parquet('{Archivo_PAR_REGIONALES_PAIS}') RO
          ON CO.ID_REGIONAL = RO.ID_REGIONAL
        LEFT JOIN ZON Z
          ON Z.ID_TIPO_ZONA_URBA = B.DOC_ID_TIPO_ZONA_URBA
         AND Z.ID_ZONA_URBA      = B.DOC_ID_ZONA_URBA
         AND Z.ID_CIUDAD         = B.DOC_ID_CIUDAD_ZONA_URBA
         AND Z.ID_PAIS = 1
        WHERE B.SEG_ID_PROCESO IN ({ids_entrega_sql})
    )
    SELECT
        *,
        CAST(TIEMPO_ENTREGA_HORAS > TIEMPO_COMERCIAL_HOR AS INTEGER) AS INCUMPLIMIENTO_SLA,
        hour(G_FECHA_ENVIO)       AS HORA_ENVIO,
        dayofweek(G_FECHA_ENVIO)  AS DIA_SEMANA,          -- 0 = domingo
        month(G_FECHA_ENVIO)      AS MES_ENVIO,
        CAST(G_FECHA_ENVIO AS DATE) AS FECHA_ENVIO_DIA,
        CASE WHEN hour(G_FECHA_ENVIO) BETWEEN 6 AND 11 THEN 'MANANA'
             WHEN hour(G_FECHA_ENVIO) BETWEEN 12 AND 17 THEN 'TARDE'
             WHEN hour(G_FECHA_ENVIO) BETWEEN 18 AND 22 THEN 'NOCHE'
             ELSE 'MADRUGADA' END  AS FRANJA_HORARIA,
        CAST(dayofweek(G_FECHA_ENVIO) IN (0, 6) AS INTEGER) AS FIN_DE_SEMANA,
        COALESCE(SEG_LARGO_ENVIO, 0) * COALESCE(SEG_ANCHO_ENVIO, 0) * COALESCE(SEG_ALTO_ENVIO, 0) AS VOLUMEN_CM3
    FROM BASE
    WHERE TIEMPO_ENTREGA_HORAS BETWEEN 0 AND 720
      AND TIEMPO_COMERCIAL_HOR IS NOT NULL
)
TO '{RUTA_BASE_FINAL}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.close()
print("BASE_FINAL_BOGOTA.parquet generada:", RUTA_BASE_FINAL)

## 3. Resumen y EDA rápido (cifras para la presentación)

In [ ]:
# CELDA 7 · Cifras generales
df_resumen = duckdb.sql(f"""
    SELECT
        COUNT(*)                                   AS ENVIOS,
        ROUND(100 * AVG(INCUMPLIMIENTO_SLA), 2)    AS TASA_INCUMPLIMIENTO_PCT,
        ROUND(MEDIAN(TIEMPO_ENTREGA_HORAS), 1)     AS TIEMPO_MEDIANO_H,
        ROUND(AVG(TIEMPO_ENTREGA_HORAS), 1)        AS TIEMPO_PROMEDIO_H,
        ROUND(MEDIAN(TIEMPO_COMERCIAL_HOR), 1)     AS SLA_MEDIANO_H,
        COUNT(DISTINCT ZONA_ID)                    AS ZONAS
    FROM read_parquet('{RUTA_BASE_FINAL}')
""").df()
display(df_resumen)

df_mes = duckdb.sql(f"""
    SELECT MES_ENVIO, COUNT(*) AS ENVIOS,
           ROUND(100 * AVG(INCUMPLIMIENTO_SLA), 2) AS TASA_INCUMPLIMIENTO_PCT,
           ROUND(MEDIAN(TIEMPO_ENTREGA_HORAS), 1)  AS TIEMPO_MEDIANO_H
    FROM read_parquet('{RUTA_BASE_FINAL}')
    GROUP BY MES_ENVIO ORDER BY MES_ENVIO
""").df()
display(df_mes)

In [ ]:
# CELDA 8 · Incumplimiento por franja, día y zona (gráficas para el EDA)
df_franja = duckdb.sql(f"""
    SELECT FRANJA_HORARIA, COUNT(*) AS ENVIOS, AVG(INCUMPLIMIENTO_SLA) AS TASA
    FROM read_parquet('{RUTA_BASE_FINAL}') GROUP BY 1 ORDER BY TASA DESC""").df()
df_dia = duckdb.sql(f"""
    SELECT DIA_SEMANA, COUNT(*) AS ENVIOS, AVG(INCUMPLIMIENTO_SLA) AS TASA
    FROM read_parquet('{RUTA_BASE_FINAL}') GROUP BY 1 ORDER BY 1""").df()
df_zona = duckdb.sql(f"""
    SELECT COALESCE(ZONA_URBANA, ZONA_ID) AS ZONA, COUNT(*) AS ENVIOS, AVG(INCUMPLIMIENTO_SLA) AS TASA,
           MEDIAN(TIEMPO_ENTREGA_HORAS) AS TIEMPO_MEDIANO_H
    FROM read_parquet('{RUTA_BASE_FINAL}') GROUP BY 1 ORDER BY ENVIOS DESC LIMIT 15""").df()

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].bar(df_franja["FRANJA_HORARIA"], df_franja["TASA"] * 100)
axes[0].set_title("Incumplimiento por franja de envío (%)")
axes[1].bar(df_dia["DIA_SEMANA"].astype(str), df_dia["TASA"] * 100)
axes[1].set_title("Incumplimiento por día (0 = domingo) (%)")
z = df_zona.sort_values("TASA")
axes[2].barh(z["ZONA"].astype(str), z["TASA"] * 100)
axes[2].set_title("Incumplimiento en las 15 zonas con más envíos (%)")
plt.tight_layout(); plt.show()
display(df_zona)

## 4. Modelado: LightGBM vs. línea base, con validación temporal y espacial

- **Escenario A · Aleatorio:** 80/20 al azar (referencia optimista).
- **Escenario B · Temporal:** entrenar con meses ≤ `MES_CORTE`, probar con los siguientes.
- **Escenario C · Temporal + espacial:** igual que B, pero las zonas de prueba **nunca** aparecen en el entrenamiento (5 grupos de zonas, `GroupKFold`).

Las variables históricas por zona y franja se calculan **solo con el entrenamiento** de cada escenario (evita fuga de información). Si el error crece mucho de A a C, hay sobreajuste.

In [ ]:
# CELDA 9 · Carga de una muestra para la primera corrida
N_MUESTRA = 500_000   # subirlo cuando todo funcione (p. ej. 2_000_000 o la base completa)

RUTA_BASE_MODELO = RUTA_BASE_FINAL.replace("BASE_FINAL_BOGOTA.parquet", "BASE_FINAL_BOGOTA_CLIMA.parquet") if USAR_CLIMA else RUTA_BASE_FINAL
COLS_CLIMA = ", LLUEVE_HORA, LLUVIA_3H_PREV, LLUVIA_24H_PREV, LLUVIA_72H_PREV, TEMPERATURA_C, HUMEDAD_PCT" if USAR_CLIMA else ""

df = duckdb.sql(f"""
    SELECT G_FECHA_ENVIO, FECHA_ENVIO_DIA, MES_ENVIO, HORA_ENVIO, DIA_SEMANA, FRANJA_HORARIA, FIN_DE_SEMANA,
           ZONA_ID, CATEGORIA_ZONA, REGIONAL_ORIGEN, G_ID_TIPOGUIA, G_ID_PRODUCTO,
           SEG_PESO_ENVIO, SEG_NUMERO_PIEZAS, VOLUMEN_CM3, TIEMPO_COMERCIAL_HOR,
           TIEMPO_ENTREGA_HORAS, INCUMPLIMIENTO_SLA {COLS_CLIMA}
    FROM read_parquet('{RUTA_BASE_MODELO}')
    USING SAMPLE {N_MUESTRA} ROWS
""").df()

try:
    import holidays
    festivos = holidays.CO(years=sorted(pd.to_datetime(df["FECHA_ENVIO_DIA"]).dt.year.unique()))
    df["ES_FESTIVO"] = pd.to_datetime(df["FECHA_ENVIO_DIA"]).dt.date.isin(festivos).astype(int)
except ImportError:
    df["ES_FESTIVO"] = 0

print(f"Muestra: {len(df):,} envíos | Tasa de incumplimiento: {df['INCUMPLIMIENTO_SLA'].mean():.1%}")
display(df["MES_ENVIO"].value_counts().sort_index())

In [ ]:
# CELDA 10 · Variables, modelos y métricas
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             precision_score, recall_score, f1_score, average_precision_score)

RANDOM_STATE = 42
CATEGORICAS = ["FRANJA_HORARIA", "CATEGORIA_ZONA", "REGIONAL_ORIGEN", "G_ID_TIPOGUIA", "G_ID_PRODUCTO"]
NUMERICAS = ["HORA_ENVIO", "DIA_SEMANA", "FIN_DE_SEMANA", "ES_FESTIVO",
             "SEG_PESO_ENVIO", "SEG_NUMERO_PIEZAS", "VOLUMEN_CM3", "TIEMPO_COMERCIAL_HOR",
             "HIST_TIEMPO_ZONA_FRANJA", "HIST_INCUMP_ZONA_FRANJA", "HIST_VOLUMEN_ZONA"]   # espacio-temporales
if USAR_CLIMA:
    NUMERICAS += ["LLUEVE_HORA", "LLUVIA_3H_PREV", "LLUVIA_24H_PREV", "LLUVIA_72H_PREV", "TEMPERATURA_C", "HUMEDAD_PCT"]
FEATURES = CATEGORICAS + NUMERICAS
for c in CATEGORICAS:
    df[c] = df[c].astype(str)

def agregar_historicos(train, test):
    """Retraso, incumplimiento y volumen históricos por zona y franja, calculados SOLO con train."""
    train, test = train.copy(), test.copy()
    g = train.groupby(["ZONA_ID", "FRANJA_HORARIA"]).agg(
        HIST_TIEMPO_ZONA_FRANJA=("TIEMPO_ENTREGA_HORAS", "mean"),
        HIST_INCUMP_ZONA_FRANJA=("INCUMPLIMIENTO_SLA", "mean")).reset_index()
    v = train.groupby("ZONA_ID").size().rename("HIST_VOLUMEN_ZONA").reset_index()
    out = []
    for d in (train, test):
        d = d.merge(g, on=["ZONA_ID", "FRANJA_HORARIA"], how="left").merge(v, on="ZONA_ID", how="left")
        d["HIST_TIEMPO_ZONA_FRANJA"] = d["HIST_TIEMPO_ZONA_FRANJA"].fillna(train["TIEMPO_ENTREGA_HORAS"].mean())
        d["HIST_INCUMP_ZONA_FRANJA"] = d["HIST_INCUMP_ZONA_FRANJA"].fillna(train["INCUMPLIMIENTO_SLA"].mean())
        d["HIST_VOLUMEN_ZONA"] = d["HIST_VOLUMEN_ZONA"].fillna(0)   # zona nunca vista
        out.append(d)
    return out[0], out[1]

def preparar_lgb(d):
    X = d[FEATURES].copy()
    for c in CATEGORICAS:
        X[c] = X[c].astype("category")
    return X

def linea_base(tarea):
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), CATEGORICAS),
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), NUMERICAS)])
    modelo = LinearRegression() if tarea == "regresion" else LogisticRegression(max_iter=500, class_weight="balanced")
    return Pipeline([("prep", prep), ("modelo", modelo)])

def entrenar_y_predecir(train, test):
    """Devuelve predicciones de los 4 modelos sobre test."""
    train, test = agregar_historicos(train, test)
    Xtr, Xte = preparar_lgb(train), preparar_lgb(test)
    # alinear categorías de test con las de train
    for c in CATEGORICAS:
        Xte[c] = pd.Categorical(Xte[c], categories=Xtr[c].cat.categories)
    pred = {}
    lgb_reg = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=63,
                                min_child_samples=100, random_state=RANDOM_STATE, verbose=-1)
    lgb_reg.fit(Xtr, train["TIEMPO_ENTREGA_HORAS"])
    pred["reg_LightGBM"] = lgb_reg.predict(Xte)
    lgb_clf = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=100,
                                 is_unbalance=True, random_state=RANDOM_STATE, verbose=-1)
    lgb_clf.fit(Xtr, train["INCUMPLIMIENTO_SLA"])
    pred["clf_LightGBM"] = lgb_clf.predict_proba(Xte)[:, 1]
    base_reg = linea_base("regresion").fit(train[FEATURES], train["TIEMPO_ENTREGA_HORAS"])
    pred["reg_Lineal"] = base_reg.predict(test[FEATURES])
    base_clf = linea_base("clasificacion").fit(train[FEATURES], train["INCUMPLIMIENTO_SLA"])
    pred["clf_Logistica"] = base_clf.predict_proba(test[FEATURES])[:, 1]
    pred["y_reg"] = test["TIEMPO_ENTREGA_HORAS"].values
    pred["y_clf"] = test["INCUMPLIMIENTO_SLA"].values
    return pred, lgb_reg, lgb_clf

def metricas(escenario, pred):
    filas = []
    for m in ["LightGBM", "Lineal"]:
        y, p = pred["y_reg"], pred[f"reg_{m}"]
        filas.append({"escenario": escenario, "tarea": "Regresión (tiempo)", "modelo": m,
                      "MAE_h": mean_absolute_error(y, p), "RMSE_h": np.sqrt(mean_squared_error(y, p))})
    for m in ["LightGBM", "Logistica"]:
        y, p = pred["y_clf"], pred[f"clf_{m}"]
        yhat = (p >= 0.5).astype(int)
        filas.append({"escenario": escenario, "tarea": "Clasificación (SLA)",
                      "modelo": "LightGBM" if m == "LightGBM" else "Logística",
                      "Recall": recall_score(y, yhat, zero_division=0),
                      "Precision": precision_score(y, yhat, zero_division=0),
                      "F1": f1_score(y, yhat, zero_division=0),
                      "AUC_PR": average_precision_score(y, p) if y.sum() > 0 else np.nan})
    return filas

print("Funciones listas. Variables del modelo:", len(FEATURES))

In [ ]:
# CELDA 11 · Correr los tres escenarios
resultados = []

# A · Aleatorio (referencia optimista)
tr, te = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE)
pred_A, _, _ = entrenar_y_predecir(tr, te)
resultados += metricas("A · Aleatorio", pred_A)
print("Escenario A listo")

# B · Temporal
tr = df[df["MES_ENVIO"] <= MES_CORTE]
te = df[df["MES_ENVIO"] > MES_CORTE]
print(f"Temporal → train: {len(tr):,} | test: {len(te):,}")
pred_B, modelo_reg_B, modelo_clf_B = entrenar_y_predecir(tr, te)
resultados += metricas("B · Temporal", pred_B)
print("Escenario B listo")

# C · Temporal + espacial (zonas de prueba nunca vistas en entrenamiento)
zonas = df["ZONA_ID"].unique()
gkf = GroupKFold(n_splits=5)
acumulado = {}
for k, (_, idx_zonas) in enumerate(gkf.split(zonas, groups=zonas)):
    zonas_test = set(zonas[idx_zonas])
    tr = df[(df["MES_ENVIO"] <= MES_CORTE) & (~df["ZONA_ID"].isin(zonas_test))]
    te = df[(df["MES_ENVIO"] > MES_CORTE) & (df["ZONA_ID"].isin(zonas_test))]
    if len(te) == 0:
        continue
    p, _, _ = entrenar_y_predecir(tr, te)
    for key, val in p.items():
        acumulado.setdefault(key, []).append(val)
    print(f"  pliegue espacial {k+1}/5 → train {len(tr):,} | test {len(te):,}")
pred_C = {k: np.concatenate(v) for k, v in acumulado.items()}
resultados += metricas("C · Temporal + espacial", pred_C)
print("Escenario C listo")

In [ ]:
# CELDA 12 · Tabla de resultados (para el documento y la presentación)
df_res = pd.DataFrame(resultados)
reg = df_res[df_res["tarea"].str.startswith("Regresión")][["escenario", "modelo", "MAE_h", "RMSE_h"]].round(2)
clf = df_res[df_res["tarea"].str.startswith("Clasificación")][["escenario", "modelo", "Recall", "Precision", "F1", "AUC_PR"]].round(3)
print("REGRESIÓN · tiempo de entrega (horas)"); display(reg)
print("CLASIFICACIÓN · incumplimiento de SLA");  display(clf)

df_res["con_clima"] = USAR_CLIMA
RUTA_RESULTADOS = RUTA_RESULTADOS.replace(".csv", "_CON_CLIMA.csv" if USAR_CLIMA else "_SIN_CLIMA.csv")
df_res.to_csv(RUTA_RESULTADOS, index=False)
print("Resultados guardados en:", RUTA_RESULTADOS)
print("Lectura: si el error de C es mucho mayor que el de A, el modelo se sobreajusta a periodos/zonas conocidos.")

In [ ]:
# CELDA 13 · Variables más importantes del LightGBM (escenario temporal)
imp = pd.DataFrame({"variable": FEATURES,
                    "importancia_regresion": modelo_reg_B.booster_.feature_importance("gain"),
                    "importancia_clasificacion": modelo_clf_B.booster_.feature_importance("gain")})
imp = imp.sort_values("importancia_clasificacion", ascending=True)
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp["variable"], imp["importancia_clasificacion"])
ax.set_title("Importancia (gain) · LightGBM clasificación · escenario temporal")
plt.tight_layout(); plt.show()

horafin = datetime.datetime.now()
print(f"Tiempo de ejecución: {horafin - horainicio}")